In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [25]:
from sklearn.model_selection import train_test_split
X=df.drop('Outcome',axis=1)
y=df['Outcome']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()

This dataset is in need of some preprocessing.
Values that are 0 in columns other than Pregnancies and Outcome are actually NaN

In [26]:
nancols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
X_train[nancols] = X_train[nancols].replace(0,np.nan)
X_train.head()
X_test[nancols] = X_test[nancols].replace(0,np.nan)
X_test.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
668,6,98,58.0,33.0,190.0,34.0,0.430,43
324,2,112,75.0,32.0,NaN,35.7,0.148,21
624,2,108,64.0,NaN,NaN,30.8,0.158,21
690,8,107,80.0,NaN,NaN,24.6,0.856,34
473,7,136,90.0,NaN,NaN,29.9,0.210,50


In [27]:
#time to impute the NaNs with the median
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

In [28]:
#scaling values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_std_lr = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train.columns, index=X_train.index)
X_test_std_lr = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test.columns, index=X_test.index)
X_train_std_knn = X_train_std_lr.copy()
X_test_std_knn = X_test_std_lr.copy()


MODEL 1: LOGISTIC REGRESSION FROM SCRATCH

In [29]:
class LogisticRegression:
  def __init__(self, lr=0.01, iterations=5000, threshold=0.5):
    self.lr = lr
    self.iterations = iterations
    self.threshold = threshold

  @staticmethod
  def sigmoid(z):
    z = np.clip(z,-500,500)
    return (1.0/(1.0 + np.exp(-z)))

  def fit(self, X, y):
    self.w = np.zeros(X.shape[1])
    self.b=0
    for i in range(self.iterations):
      z = X@self.w + self.b
      p = self.sigmoid(z)
      error = p - y
      grad_w = (X.T@error)/X.shape[0]
      grad_b = error.mean()
      self.w = self.w - self.lr*grad_w
      self.b = self.b - self.lr*grad_b
      cost_function = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
      if(i%10==0):
        print(f"Iteration {i} : Cost Function : {cost_function}")
    return self
  def predict(self,X):
    return (self.sigmoid(X@self.w  +self.b) >= self.threshold).astype(int)

In [31]:
y_pred = LogisticRegression(0.05,10000).fit(X_train_std_lr,y_train).predict(X_test_std_lr)

Iteration 0 : Cost Function : 0.6931471805599453
Iteration 10 : Cost Function : 0.6300261566970795
Iteration 20 : Cost Function : 0.5900042004990463
Iteration 30 : Cost Function : 0.5632507343894727
Iteration 40 : Cost Function : 0.544426453049504
Iteration 50 : Cost Function : 0.5305897700804222
Iteration 60 : Cost Function : 0.520049527801397
Iteration 70 : Cost Function : 0.5117854503059018
Iteration 80 : Cost Function : 0.5051530662917356
Iteration 90 : Cost Function : 0.4997280855712699
Iteration 100 : Cost Function : 0.4952207979658583
Iteration 110 : Cost Function : 0.4914269800995418
Iteration 120 : Cost Function : 0.488198618127945
Iteration 130 : Cost Function : 0.4854258128423432
Iteration 140 : Cost Function : 0.48302522787388646
Iteration 150 : Cost Function : 0.4809324958552222
Iteration 160 : Cost Function : 0.47909709258043476
Iteration 170 : Cost Function : 0.4774787933256968
Iteration 180 : Cost Function : 0.47604516943222214
Iteration 190 : Cost Function : 0.47476978

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def metrics_row(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
    }
print(metrics_row("Logistic Regression (scratch)", y_test, y_pred))

{'Model': 'Logistic Regression (scratch)', 'Accuracy': 0.7532467532467533, 'Precision': 0.6666666666666666, 'Recall': 0.6181818181818182, 'F1': 0.6415094339622641}


MODEL 2: K-NN scratch

In [33]:
class KNearestNeighbours:
  def __init__(self,k=5):
    self.k = k
  def fit(self,X,y):
    self.X_train = X
    self.y_train = y.astype(int) # Convert y to integer type
    return self
  def predict(self,X):
    preds = np.empty(X.shape[0],dtype=int)
    for i,x in enumerate(X):
      dists = np.sqrt(np.sum((self.X_train-x)**2,axis=1))
      nearest_dists_index = np.argsort(dists)[:self.k]
      nearest_labels = self.y_train[nearest_dists_index]
      bruh = np.bincount(nearest_labels)
      preds[i] = np.argmax(bruh)
    return preds

In [34]:
y_pred = KNearestNeighbours(4).fit(X_train_std_knn.values, y_train.values).predict(X_test_std_knn.values)

In [35]:
print(metrics_row("KNN (scratch)" , y_test, y_pred))

{'Model': 'KNN (scratch)', 'Accuracy': 0.7337662337662337, 'Precision': 0.6521739130434783, 'Recall': 0.5454545454545454, 'F1': 0.594059405940594}


MODEL 3 : GAUSSIAN NAIVE BAYES FROM SCRATCH


In [36]:
class GaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        n_features = X.shape[1]
        self.mean = {}
        self.var = {}
        self.priors = {}
        for c in self.classes:
            Xc = X[y == c]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c] = Xc.var(axis=0) + 1e-9
            self.priors[c] = Xc.shape[0] / X.shape[0]
        return self

    def _log_gaussian(self, X, mean, var):
        return -0.5 * np.log(2 * np.pi * var) - ((X - mean) ** 2) / (2 * var)

    def predict_log_proba(self, X):
        log_probs = np.zeros((X.shape[0], len(self.classes)))
        for idx, c in enumerate(self.classes):
            log_likelihood = self._log_gaussian(X, self.mean[c], self.var[c]).sum(axis=1)
            log_probs[:, idx] = np.log(self.priors[c]) + log_likelihood
        return log_probs

    def predict(self, X):
        log_probs = self.predict_log_proba(X)
        return self.classes[np.argmax(log_probs, axis=1)]

In [37]:
y_pred = GaussianNB().fit(X_train.values, y_train.values).predict(X_test.values)

In [38]:
print(metrics_row("Naive Bayes (scratch)" , y_test, y_pred))

{'Model': 'Naive Bayes (scratch)', 'Accuracy': 0.6428571428571429, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


We can see that the priors part is dominated by a particular class , hence the model always selects the dominating class